In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install catboost vmdpy -q
!cd /content/drive/MyDrive/RuzgarTahmin && unzip -oq ruzgar_sizinti_paketi.zip
!ls /content/drive/MyDrive/RuzgarTahmin/ruzgar_sizinti_paketi

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00
leakage  notebooks  README.md  requirements.txt  scripts  tests


In [4]:
import os
os.environ['RT_VERI_CSV'] = '/content/drive/MyDrive/RuzgarTahmin/veri/sdwpf_2001_2112_full.csv'
os.environ['RT_OUT'] = '/content/drive/MyDrive/RuzgarTahmin/yeni/'
%cd /content/drive/MyDrive/RuzgarTahmin/ruzgar_sizinti_paketi
!python scripts/06_maliyet_olcumu.py

/content/drive/MyDrive/RuzgarTahmin/ruzgar_sizinti_paketi
Donanım: x86_64 | CPU çekirdek: 2 | Python 3.13.15
Tüm seri (35,089 adım): LCSF 0.004 sn | Full VMD 17.1 sn | Rolling VMD ≈ 17 dk (tek çekirdek)
Çevrimiçi gecikme: LCSF 56 µs | Rolling VMD 29.3 ms | oran ≈ 526×


In [5]:
  !lscpu | grep -E "Model name|^CPU\(s\)"

CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz


In [6]:
import os, shutil
os.environ['RT_VERI_CSV'] = '/content/drive/MyDrive/RuzgarTahmin/veri/sdwpf_2001_2112_full.csv'
os.environ['RT_OUT'] = '/content/drive/MyDrive/RuzgarTahmin/yeni/'
%cd /content/drive/MyDrive/RuzgarTahmin/ruzgar_sizinti_paketi
OUT = os.environ['RT_OUT']
shutil.copy(OUT+'adim6_T10.pkl', OUT+'adim6_T10_ESKI_yedek.pkl')
print('yedek alındı')

/content/drive/MyDrive/RuzgarTahmin/ruzgar_sizinti_paketi
yedek alındı


In [7]:
!python scripts/02_turbin_kosusu.py 10 2>&1 | grep -v -i -E "cuda|tensorflow/|oneDNN|cpu_feature|AVX"

T10: N=35,089 | eğitim/doğrulama/test = 24,562/5,251/5,252 | frekanslar=[0.0006 0.0187 0.0434 0.0886]
   T10 H=1.0 sa modeller tamam | 6.8 dk

Türbin 10 | H=1.0 sa
  Persistence                      MAE=  110.58  Skill=   0.00%
  Raw LSTM                         MAE=  131.73  Skill= -19.13%
  CatBoost Direct                  MAE=  116.72  Skill=  -5.56%
  Full VMD-LSTM (sızdıran)         MAE=   69.58  Skill=  37.07%
  Rolling VMD-LSTM (nedensel)      MAE=  128.17  Skill= -15.91%
  LCSF çift yönlü (sızdıran)       MAE=   55.57  Skill=  49.74%
  LCSF nedensel                    MAE=  127.71  Skill= -15.49%
  B9 LCSF+OOF CatBoost             MAE=  114.26  Skill=  -3.33%
  LSI VMD  = 45.7%  |  LSI LCSF = 56.5%
  DM VMD sızıntı: -27.82, p=0.00e+00
  DM LCSF sızıntı: -33.25, p=0.00e+00


In [8]:
import pickle, numpy as np
eski = pickle.load(open(OUT+'adim6_T10_ESKI_yedek.pkl','rb'))
yeni = pickle.load(open(OUT+'adim6_T10.pkl','rb'))
for k in ['B1','B2','B3','B6','B7','B8','B8L','B9']:
    a, b = eski[(k,2)], yeni[(k,2)]
    ayni_t = np.array_equal(a['t'], b['t'])
    mae_e = np.mean(np.abs(a['y']-a['p'])); mae_y = np.mean(np.abs(b['y']-b['p']))
    print(f"{k:4s} test kümesi aynı: {ayni_t} | MAE eski {mae_e:8.2f}  yeni {mae_y:8.2f}  fark {100*(mae_y-mae_e)/mae_e:+.2f}%  | maks tahmin farkı {np.abs(a['p']-b['p']).max():.3f}")

B1   test kümesi aynı: True | MAE eski   110.58  yeni   110.58  fark +0.00%  | maks tahmin farkı 0.000
B2   test kümesi aynı: True | MAE eski   131.73  yeni   131.73  fark +0.00%  | maks tahmin farkı 0.000
B3   test kümesi aynı: True | MAE eski   116.72  yeni   116.72  fark +0.00%  | maks tahmin farkı 0.000
B6   test kümesi aynı: True | MAE eski    69.58  yeni    69.58  fark +0.00%  | maks tahmin farkı 0.000
B7   test kümesi aynı: True | MAE eski   128.17  yeni   128.17  fark +0.00%  | maks tahmin farkı 0.000
B8   test kümesi aynı: True | MAE eski   127.71  yeni   127.71  fark +0.00%  | maks tahmin farkı 0.000
B8L  test kümesi aynı: True | MAE eski    55.57  yeni    55.57  fark +0.00%  | maks tahmin farkı 0.000
B9   test kümesi aynı: True | MAE eski   114.26  yeni   114.26  fark +0.00%  | maks tahmin farkı 0.000


In [9]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/RuzgarTahmin/veri/sdwpf_2001_2112_full.csv')
print(df.groupby('TurbID')['Patv'].max().loc[[57,10,83,120,61,84]])
df['yil'] = pd.to_datetime(df['Tmstamp']).dt.year
print(df[df.TurbID==57].assign(t=lambda x: pd.to_datetime(x.Tmstamp)).groupby('yil')['t'].apply(lambda s: s.diff().dropna().value_counts().head(3)))

TurbID
57     1521.49
10     1560.65
83     1561.38
120    1521.42
61     1520.97
84     1520.89
Name: Patv, dtype: float64
yil                  
2020  0 days 00:30:00    29816
      0 days 01:00:00      811
      0 days 01:30:00      278
2021  0 days 00:10:00    52558
2022  0 days 00:10:00        1
Name: t, dtype: int64
